# Resource Selection Function

In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import ee

from movement_models import (
    FeatureSpec,
    _build_design_matrix,
    _get_availability_domain,
    _get_sampling_points,
    _sample_env_layer,
    load_presence_csvs,
    to_reloc_gdf,
    reproject_reloc,
    to_reloc_gdf_projected,
    init_ee,
    init_ee_on_client,
    make_aoi,
    aoi_to_ee,
    build_predictors_image,
    ee_image_to_env_xarray,
    save_env_zarr,
    load_env_zarr,
    shutdown_default_client,
    make_local_dask_client,
    
    fit_rsf,
    predict_rsf_points,
    get_rsf_surface,
    fixed_width_Boyce,
    sliding_window_Boyce,
    cv_model,
    eval_all_linear_candidates
)

In [ ]:
def _choose_chunk_size_points(env, *, client=None, frac_of_worker_mem=0.10, per_worker_budget_mb=None, k=10,
    min_points=1_000, max_points=200_000):

    if client is not None:
        info = client.scheduler_info()["workers"]
        limits_mb = [w["memory_limit"] / 1024**2 for w in info.values()]
        base_budget_mb = min(limits_mb) * frac_of_worker_mem
        per_worker_budget_mb = int(max(8, min(512, base_budget_mb)))
    else:
        if per_worker_budget_mb is None:
            per_worker_budget_mb = 16  # conservative fallback
        per_worker_budget_mb = int(per_worker_budget_mb)

    if hasattr(env, "dims") and "band" in env.dims:
        n_bands = int(env.sizes["band"])
        dtype = env.dtype
    else:
        n_bands = len(env.data_vars)
        dtype = next(iter(env.data_vars.values())).dtype

    bytes_per_value = np.dtype(dtype).itemsize
    budget_bytes = per_worker_budget_mb * 1024 * 1024

    bytes_per_point = n_bands * bytes_per_value
    n_points = budget_bytes // (k * bytes_per_point)

    return int(min(max_points, max(min_points, n_points)))

In [ ]:
def _sample_env_layer_old(
    samples: gpd.GeoDataFrame,
    env: xr.DataArray,
    chunk_size_points="auto",
    *,
    client=None,
    per_worker_budget_mb=16,
    k=10,
    min_points=1_000,
    max_points=200_000,
):
    try:
        env_crs = env.rio.crs
    except Exception:
        env_crs = None

    if env_crs is None:
        raise ValueError("env has no rio CRS; set it with env = env.rio.write_crs('EPSG:...')")

    if samples.crs is None:
        raise ValueError("samples.crs is None; set a CRS on your GeoDataFrame before sampling")

    if samples.crs != env_crs:
        samples = samples.to_crs(env_crs)

    if chunk_size_points == "auto" or chunk_size_points is None:
        chunk_size_points = _choose_chunk_size_points(
            env,
            client=client,
            per_worker_budget_mb=per_worker_budget_mb,
            k=k,
            min_points=min_points,
            max_points=max_points,
        )
    else:
        chunk_size_points = int(chunk_size_points)

    xs = xr.DataArray(samples.geometry.x.to_numpy(), dims="points", name="x")
    ys = xr.DataArray(samples.geometry.y.to_numpy(), dims="points", name="y")

    sampled = env.sel(x=xs, y=ys, method="nearest").chunk({"points": chunk_size_points})

    arr = sampled.data
    arr = arr.compute() if hasattr(arr, "compute") else np.asarray(arr)
    arr = np.asarray(arr, dtype="float32")

    bands = sampled["band"].to_numpy().tolist()
    df = pd.DataFrame(arr.T, columns=bands)
    df["x"] = xs.to_numpy()
    df["y"] = ys.to_numpy()
    df["used"] = samples["used"].to_numpy()
    df["Timestamp"] = samples["Timestamp"].to_numpy()
    return df

In [ ]:
import importlib
import movement_models.sampling as smp
importlib.reload(smp)

_sample_env_layer_new = smp._sample_env_layer

In [ ]:
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]
cov = env.sel(band=subset_predictors)

domain = _get_availability_domain(reloc)
samples = _get_sampling_points(domain, 10_000, df=reloc.iloc[:1000], seed=1)

In [ ]:
from time import perf_counter

t0 = perf_counter()
out_old = _sample_env_layer_old(samples, cov, chunk_size_points="auto", client=client)
t1 = perf_counter()

t2 = perf_counter()
out_new = _sample_env_layer(samples, cov, chunk_size_points="auto", client=client)
t3 = perf_counter()

print("old runtime:", t1 - t0)
print("new runtime:", t3 - t2)

In [ ]:
print(out_old.shape, out_new.shape)
print(list(out_old.columns))
print(list(out_new.columns))

In [ ]:
import numpy as np

sort_cols = ["Timestamp", "used", "x", "y"]

a = out_old.sort_values(sort_cols).reset_index(drop=True)
b = out_new.sort_values(sort_cols).reset_index(drop=True)

print((a["used"].values == b["used"].values).all())
print((a["Timestamp"].values == b["Timestamp"].values).all())

float_cols = ["x", "y"] + subset_predictors

for col in float_cols:
    ok = np.allclose(a[col].to_numpy(), b[col].to_numpy(), equal_nan=True, rtol=1e-7, atol=1e-8)
    max_diff = np.nanmax(np.abs(a[col].to_numpy() - b[col].to_numpy()))
    print(col, ok, max_diff)

In [ ]:
spec = FeatureSpec(linear=subset_predictors, add_const=True)

m_old, scaler_old, _ = fit_rsf(out_old, spec)
pred_old = predict_rsf_points(out_old, m_old, scaler_old, spec)
rsf_old = get_rsf_surface(cov, m_old, scaler_old, spec, crs=cov.rio.crs)
B_old, _ = fixed_width_Boyce(pred_old, rsf_old, domain, seed=1)

m_new, scaler_new, _ = fit_rsf(out_new, spec)
pred_new = predict_rsf_points(out_new, m_new, scaler_new, spec)
rsf_new = get_rsf_surface(cov, m_new, scaler_new, spec, crs=cov.rio.crs)
B_new, _ = fixed_width_Boyce(pred_new, rsf_new, domain, seed=1)

print("Boyce old:", B_old)
print("Boyce new:", B_new)

In [ ]:
client = make_local_dask_client(memory_limit="2GB") 

init_status = init_ee_on_client(client, project_id="ee-alvykabo")
client, init_status

## 1. Preparation

### 1.1 Fetch presence data

In [ ]:
reloc_df = load_presence_csvs("data/*.csv")
reloc = to_reloc_gdf(reloc_df)

reloc.head(), reloc.crs, reloc["Timestamp"].dtype

### 1.2 Fetch environmental layers

In [ ]:
ee.Authenticate()
init_ee(project_id="ee-alvykabo")   

aoi = make_aoi(reloc, buffer_m=1)
aoi_ee = aoi_to_ee(aoi)

predictors_img = build_predictors_image(aoi_ee, start="2024-01-01", end="2024-12-31")
env = ee_image_to_env_xarray(predictors_img, aoi_ee, crs="EPSG:29333", scale=100, chunk_xy=1024)

save_env_zarr(env, "env_29333.zarr", mode="w")
env2 = load_env_zarr("env_29333.zarr")

env.shape, env.rio.crs, env2.shape

## 2. Calculate the RSF

### 2.1 Define the sampling scheme

In [ ]:
domain = _get_availability_domain(reloc)
samples = _get_sampling_points(domain, 1000, df=reloc.iloc[:100], seed=1)
sampled = _sample_env_layer(samples, env.sel(band=["ndvi", "slope"]), chunk_size_points="auto", client=client)

sampled.head()

### 2.2 Extract values from the environmental covariates

### 2.3 Fit logistic regression

In [ ]:
spec = FeatureSpec(linear=["ndvi", "slope"], add_const=True)
m, scaler, _ = fit_rsf(sampled, spec)
pred = predict_rsf_points(sampled, m, scaler, spec)

pred[["rsf_pred"]].describe()

### 2.4 Get RSF surface

In [ ]:
rsf = get_rsf_surface(env.sel(band=spec.linear), m, scaler, spec)

rsf

In [ ]:
rsf = get_rsf_surface(env, m, scaler, spec)
rsf = rsf.compute()
rsf.rio.to_raster("rsf.tif", compress="LZW")
rsf = rsf.rio.write_crs("EPSG:29333") 

In [ ]:
client.scheduler_info()["workers"].keys()
len(client.scheduler_info()["workers"])

In [ ]:
import pandas as pd

info = client.scheduler_info()["workers"]

df = pd.DataFrame({
    w: {
        "memory": data["metrics"]["memory"],
        "memory_limit": data["memory_limit"]
    }
    for w, data in info.items()
}).T

df

In [ ]:
df["memory_MB"] = df["memory"] / 1e6
df["limit_MB"] = df["memory_limit"] / 1e6
df[["memory_MB", "limit_MB"]]

### 2.5 Evaluate Using Boyce Index

In [ ]:
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]
cov = env.sel(band=subset_predictors)

samples = _get_sampling_points(domain, n=10_000, df=reloc.iloc[:1000], seed=1)
sampled = _sample_env_layer(samples, cov, chunk_size_points="auto", client=client)

spec = FeatureSpec(linear=subset_predictors, add_const=True)
m, scaler, spec = fit_rsf(sampled, spec)

pred = predict_rsf_points(sampled, m, scaler, spec)
rsf = get_rsf_surface(env.sel(band=subset_predictors), m, scaler, spec)

B, boyce = fixed_width_Boyce(pred, rsf, domain, seed=1)
B, boyce.head()

In [ ]:
B, chart = sliding_window_Boyce(
    pred=pred,
    rsf=rsf,
    domain=domain,
    window_frac=0.2,
    step_frac=0.05,
    seed=1,
)

print("B =", B)
print(chart.head())
print("rows:", len(chart), "finite pe:", chart["pe"].notna().sum())

## 3. Aggregate functions

### 3.1 Compare all possible linear models via AIC & BIC

In [ ]:
df_small = sampled.sample(n=min(2000, len(sampled)), random_state=1).copy()
subset_predictors = ["ndvi", "ndwi", "slope", "dist2water"]

df_small = df_small.replace([float("inf"), float("-inf")], pd.NA).dropna(subset=["used"] + subset_predictors)

res = eval_all_linear_candidates(df_small, env, subset=subset_predictors)
res.head()

### 3.2 Cross-Validation on a Single Individual

In [ ]:
res = cv_model(
    obs=reloc,
    env=env,
    k_folds=2,
    subset_predictors=["ndvi", "ndwi", "slope", "dist2water"],
    sampling_factor_train=3,
    n_bg_boyce=5_000,
    seed=1,
)
res

# Step, Length, & Turning Angle

In [1]:
#testing movement_kernel.py with likelihood-based fitting

from movement_models.movement import (
    prepare_trajectory_data,
    build_step_data,
    fit_step_distribution,
    build_turn_angle_data,
    fit_turn_angle_distribution,
    fit_movement_kernel_per_id,
)

from movement_models.io import (
    load_presence_csvs,
    to_reloc_gdf_projected,
)

from movement_models.io import load_presence_csvs
from movement_models.movement import fit_movement_kernel_per_id

reloc = load_presence_csvs("data/*.csv")

out = fit_movement_kernel_per_id(
    reloc,
    id_col="ID",
    timestamp_col="Timestamp",
    lon_col="Longitude",
    lat_col="Latitude",
    target_crs="EPSG:29333",
    expected_interval_min=120,
    tolerance_min=2,
    step_cutoff=20000,
)

out["summary"]

,ID,n_steps,step_distribution,step_params,step_q25,step_median,step_mean,step_q75,step_max,n_angles,vonmises_kappa,mixture_kappa,mixture_w
0,NPL38,2584,lognorm,"[2.3680072661020657, 76.329595410113]",8.809685,63.692252,546.335874,760.120739,7014.598776,2252,0.135231,6.79407,0.103583


In [2]:
step_df = out["step_df"]
one_id = step_df["ID"].dropna().iloc[0]
step_subset = step_df.loc[step_df["ID"] == one_id, "step_m"]

fit_step_distribution(step_subset)["model_table"]

,distribution,params,loglik,AIC,success
0,lognorm,"[2.3680072661020657, 76.329595410113]",-17095.920956,34195.841913,True
1,weibull,"[0.4683982286268443, 249.18223047700394]",-17177.628049,34359.256097,True
2,gamma,"[0.3428314692397524, 1593.5262762864115]",-17250.213319,34504.426638,True
3,exp,[546.3358741887124],-18871.556503,37745.113006,True
